# 03 — Comparação Líquida de IR e Equivalência de Taxas

Compara CDB, LCI/LCA, Tesouro Direto e Debêntures (comuns e incentivadas pela
Lei 12.431/2011) considerando a tabela regressiva de IR, as convenções linear e
composta de anualização líquida e a equação de Fisher (Prefixado ↔ IPCA+).

In [1]:
from rfquant.comparison import ir, taxa_equivalente

In [2]:
# 1) Tabela regressiva de IR + IOF encadeados (rendimento bruto de R$ 1.000,00)
for prazo in [15, 90, 250, 500, 800]:
    trib = ir.calcular(rendimento_bruto=1000.0, prazo_dias=prazo, isento=False)
    print(
        f"Prazo {prazo:>4} d → alíquota IR {trib.aliquota_ir:>5.1%} | "
        f"IOF R$ {trib.iof:>6.2f} | IR R$ {trib.ir:>6.2f} | "
        f"Líquido R$ {trib.rendimento_liquido:>6.2f}"
    )

Prazo   15 d → alíquota IR 22.5% | IOF R$ 500.00 | IR R$ 112.50 | Líquido R$ 387.50
Prazo   90 d → alíquota IR 22.5% | IOF R$   0.00 | IR R$ 225.00 | Líquido R$ 775.00
Prazo  250 d → alíquota IR 20.0% | IOF R$   0.00 | IR R$ 200.00 | Líquido R$ 800.00
Prazo  500 d → alíquota IR 17.5% | IOF R$   0.00 | IR R$ 175.00 | Líquido R$ 825.00
Prazo  800 d → alíquota IR 15.0% | IOF R$   0.00 | IR R$ 150.00 | Líquido R$ 850.00


In [3]:
# 2) Comparação entre produtos com isenção derivada automaticamente por `categoria`
produtos = [
    taxa_equivalente.ProdutoRF("CDB 12%", 0.12, 720, categoria="CDB"),
    taxa_equivalente.ProdutoRF("LCI 10,5%", 0.105, 720, categoria="LCI"),
    taxa_equivalente.ProdutoRF("Tesouro Prefixado 11,8%", 0.118, 720),
    taxa_equivalente.ProdutoRF("Debênture incentivada 10%", 0.10, 720, categoria="debênture-incentivada"),
]

print("--- Convenção Linear (aproximação padrão de mercado) ---")
for r in taxa_equivalente.comparar(produtos, convencao="linear"):
    print(
        f"{r.produto.nome:<28} bruto {r.produto.taxa_anual:.4%} → "
        f"líquido {r.taxa_liquida_anual:.4%} (equiv. CDB {r.equivalente_cdb:.4%})"
    )

print("\n--- Convenção Composta (exata para horizonte != 1 ano, IR único no resgate) ---")
for r in taxa_equivalente.comparar(produtos, convencao="composta"):
    print(
        f"{r.produto.nome:<28} bruto {r.produto.taxa_anual:.4%} → "
        f"líquido {r.taxa_liquida_anual:.4%} (equiv. CDB {r.equivalente_cdb:.4%})"
    )

--- Convenção Linear (aproximação padrão de mercado) ---
LCI 10,5%                    bruto 10.5000% → líquido 10.5000% (equiv. CDB 12.7273%)
Debênture incentivada 10%    bruto 10.0000% → líquido 10.0000% (equiv. CDB 12.1212%)
CDB 12%                      bruto 12.0000% → líquido 9.9000% (equiv. CDB 12.0000%)
Tesouro Prefixado 11,8%      bruto 11.8000% → líquido 9.7350% (equiv. CDB 11.8000%)

--- Convenção Composta (exata para horizonte != 1 ano, IR único no resgate) ---
LCI 10,5%                    bruto 10.5000% → líquido 10.5000% (equiv. CDB 12.7273%)
Debênture incentivada 10%    bruto 10.0000% → líquido 10.0000% (equiv. CDB 12.1212%)
CDB 12%                      bruto 12.0000% → líquido 9.9920% (equiv. CDB 12.0000%)
Tesouro Prefixado 11,8%      bruto 11.8000% → líquido 9.8241% (equiv. CDB 11.8000%)


In [4]:
# 3) Equivalências: Isento ↔ Tributado (gross-up de IR) e Prefixado ↔ IPCA+ (Fisher exata)
for taxa_lci in [0.08, 0.09, 0.10, 0.11]:
    cdb_equiv = taxa_equivalente.isento_equivalente_a_tributado(taxa_lci, 720)
    print(f"LCI {taxa_lci:.2%} ≡ CDB {cdb_equiv:.2%} (prazo 720 dias)")

ipca_proj = 0.045
real_equiv = taxa_equivalente.pre_para_ipca_mais(0.12, ipca_proj)
print(f"\nPrefixado 12,00% a.a. com IPCA projetado de {ipca_proj:.2%} ≡ IPCA + {real_equiv:.4%} a.a.")

LCI 8.00% ≡ CDB 9.70% (prazo 720 dias)
LCI 9.00% ≡ CDB 10.91% (prazo 720 dias)
LCI 10.00% ≡ CDB 12.12% (prazo 720 dias)
LCI 11.00% ≡ CDB 13.33% (prazo 720 dias)

Prefixado 12,00% a.a. com IPCA projetado de 4.50% ≡ IPCA + 7.1770% a.a.
